# Lab 5: NVIDIA CUDA with Python

Run each cell with **Shift + Enter**. Read the code first, then run it, then write your results in the lab manual.

The terminal steps (`nvcc`, `jtop`, `nvpmodel`) are in Part A and Part F of the manual. Everything here is Python.

## Part B, Step 2: Check That CuPy Works
You should see a version number and `[0 2 4 6 8]`. The first run can take several seconds.

In [ ]:
import cupy as cp
print(cp.__version__)
print(cp.arange(5) * 2)

## Part B, Step 3: Read the GPU Properties

In [ ]:
import cupy as cp

props = cp.cuda.runtime.getDeviceProperties(0)
name = props["name"]
if isinstance(name, bytes):
    name = name.decode()

print("CuPy version:", cp.__version__)
print("GPU count:", cp.cuda.runtime.getDeviceCount())
print("Name:", name)
print("Compute capability:", props["major"], props["minor"])
print("Multiprocessors (SMs):", props["multiProcessorCount"])
print("Max threads per block:", props["maxThreadsPerBlock"])
print("Warp size:", props["warpSize"])
free, total = cp.cuda.runtime.memGetInfo()
print(f"Free / total memory: {free / 1024**3:.2f} / {total / 1024**3:.2f} GB")

## Part C, Step 1: Same Code, Different Library
The two results look the same, but the second was calculated on the GPU. Run the last line to see the type.

In [ ]:
import numpy as np
import cupy as cp
a = np.arange(10)
g = cp.arange(10)
print(a * 2)
print(g * 2)
type(g)
g.shape, g.dtype

## Part C, Step 2: Copy Data to the GPU and Back
Run this cell **twice** and record the second run.

In [ ]:
import time
import numpy as np
import cupy as cp

n = 4096
a_cpu = np.random.rand(n, n).astype(np.float32)

# Warm-up: the first GPU call loads libraries
_ = cp.asarray(a_cpu)
cp.cuda.Device().synchronize()

start = time.perf_counter()
a_gpu = cp.asarray(a_cpu)
cp.cuda.Device().synchronize()
copy_ms = (time.perf_counter() - start) * 1000

start = time.perf_counter()
b_gpu = a_gpu * 2.0 + 1.0
cp.cuda.Device().synchronize()
calc_ms = (time.perf_counter() - start) * 1000

start = time.perf_counter()
b_cpu = cp.asnumpy(b_gpu)
back_ms = (time.perf_counter() - start) * 1000

print(f"Copy to GPU : {copy_ms:.2f} ms")
print(f"Calculation : {calc_ms:.2f} ms")
print(f"Copy back   : {back_ms:.2f} ms")

## Part C, Step 3: Where the GPU Wins and Loses
This times the same calculation for four array sizes on the CPU and on the GPU.

In [ ]:
import time
import numpy as np
import cupy as cp


def time_cpu(x, repeats):
    start = time.perf_counter()
    for _ in range(repeats):
        y = x * 2.0 + 1.0
    return (time.perf_counter() - start) / repeats * 1000


def time_gpu(x, repeats):
    y = x * 2.0 + 1.0  # warm-up
    cp.cuda.Device().synchronize()
    start = time.perf_counter()
    for _ in range(repeats):
        y = x * 2.0 + 1.0
    cp.cuda.Device().synchronize()
    return (time.perf_counter() - start) / repeats * 1000


for n in [100, 10_000, 1_000_000, 10_000_000]:
    x_cpu = np.random.rand(n).astype(np.float32)
    x_gpu = cp.asarray(x_cpu)
    cpu_ms = time_cpu(x_cpu, 20)
    gpu_ms = time_gpu(x_gpu, 20)
    print(f"{n:>10,} items   CPU {cpu_ms:8.3f} ms   GPU {gpu_ms:8.3f} ms")

## Part D, Step 1: An Element-Wise Kernel

In [ ]:
import cupy as cp

add_squares = cp.ElementwiseKernel(
    "float32 x, float32 y",    # inputs
    "float32 z",               # output
    "z = x * x + y * y",       # runs once for every element
    "add_squares",
)

x = cp.arange(10, dtype=cp.float32)
y = cp.ones(10, dtype=cp.float32)
z = add_squares(x, y)
print(z)

## Part D, Step 2: A Raw CUDA Kernel with Threads and Blocks
Expected: 256 threads per block, **3907** blocks and `Check: True`. The first run takes a few seconds because CuPy compiles the kernel.

In [ ]:
import cupy as cp

code = r'''
extern "C" __global__
void add_arrays(const float* a, const float* b, float* out, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        out[i] = a[i] + b[i];
    }
}
'''
add_arrays = cp.RawKernel(code, "add_arrays")

n = 1_000_000
a = cp.random.rand(n, dtype=cp.float32)
b = cp.random.rand(n, dtype=cp.float32)
out = cp.empty_like(a)

threads_per_block = 256
blocks = (n + threads_per_block - 1) // threads_per_block
add_arrays((blocks,), (threads_per_block,), (a, b, out, cp.int32(n)))
cp.cuda.Device().synchronize()

print("Threads per block:", threads_per_block)
print("Blocks:", blocks)
print("Check:", bool(cp.allclose(out, a + b)))

### Try It Yourself
- Change `threads_per_block` to 128 and then 512 in the cell above. How many blocks do you get? Is `Check` still `True`?
- Change the kernel so it calculates `out[i] = a[i] * b[i]`, and change the check to match.

In [ ]:
# Your own experiment here

## Part E, Step 1: Check That PyTorch Sees the GPU
If `CUDA available` is `False`, or the version ends with `+cpu`, do **not** run `pip install torch`. Call your instructor.

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("Compute capability:", torch.cuda.get_device_capability(0))

## Part E, Step 2: Tensors on the CPU and the GPU
`x.device` is `cpu` and `y.device` is `cuda:0`. `.cpu()` copies a tensor back.

In [ ]:
import torch
x = torch.rand(3, 3)
print(x.device)
y = x.to("cuda")
print(y.device)
z = torch.rand(3, 3, device="cuda")
print(y + z)
print((y + z).cpu())

## Part F, Step 2: Time the Matrix Multiplication with CuPy
Check the power mode in the terminal first (`sudo nvpmodel -q`). Run this cell **twice** and record the second run. Open `jtop` in a terminal to watch the GPU load.

In [ ]:
import time
import cupy as cp

sizes = [256, 512, 1024, 2048]

# Warm-up: the first GPU call loads libraries and prepares the GPU
w = cp.random.rand(64, 64, dtype=cp.float32)
_ = w @ w
cp.cuda.Device().synchronize()

for n in sizes:
    a = cp.random.rand(n, n, dtype=cp.float32)
    b = cp.random.rand(n, n, dtype=cp.float32)
    cp.cuda.Device().synchronize()

    times = []
    for _ in range(5):
        start = time.perf_counter()
        c = a @ b
        cp.cuda.Device().synchronize()
        times.append(time.perf_counter() - start)
    print(f"{n} x {n} matrix multiply: {min(times) * 1000:.2f} ms")

## Part F, Step 3: Time the Same Test with PyTorch

In [ ]:
import time
import torch

device = "cuda"
sizes = [256, 512, 1024, 2048]

# Warm-up
w = torch.rand(64, 64, device=device)
_ = w @ w
torch.cuda.synchronize()

for n in sizes:
    a = torch.rand(n, n, device=device)
    b = torch.rand(n, n, device=device)
    torch.cuda.synchronize()

    times = []
    for _ in range(5):
        start = time.perf_counter()
        c = a @ b
        torch.cuda.synchronize()
        times.append(time.perf_counter() - start)
    print(f"{n} x {n} matrix multiply: {min(times) * 1000:.2f} ms")

## Challenge: What If You Forget to Synchronise?
Copy the CuPy timing cell into the cell below. Delete the line `cp.cuda.Device().synchronize()` inside the timing loop (right after `c = a @ b`). Run it. What happens to the times, and why can you not trust them?

In [ ]:
# Your challenge cell here